# 14 - Analog-propagation baseline (C1 / C2 / C3)

**Run only after notebook 13 reports a protocol-valid Gate A.** The notebook stops otherwise (a C1-only diagnostic is possible with `ALLOW_PROTOCOL_INVALID_DIAGNOSTIC = True`; its runs are recorded as `PRELIMINARY` and never enter the leaderboard).

**Objective.** First spectrum-informed ranking baseline on the hidden-like regimes, measured with the competition metric (MRR@25, per molecule).

**Hypothesis.** For a query whose structure has no reference spectrum (C2), the most similar reference spectra (*analogs*) come from structurally similar molecules; propagating Tanimoto similarity from analog structures to each mass-compatible candidate should rank the truth above mass-only ordering.

**Method.**
1. analog neighbors: fragment + neutral-loss binned cosine over the frozen reference library, modified-cosine rescoring; excluded: the query itself, T1 duplicates, **every reference of the fold's hidden C2/C3 connectivities**;
2. candidates: universe mass index at `analog.candidate_ppm`, fold view (C3 truths removed);
3. features: Tanimoto propagation from analog molecules;
4. evaluation: heuristic scorers + LightGBM lambdarank (connectivity-grouped folds, out-of-fold only), per spectrum and **per molecule** (late fusion: mean / max / log-sum-exp / rank fusion / vote, compared -- none is assumed best), per-fold stability and the KEEP / REJECT rule (`casmi.ranking.fold_stability`).

**Caches** are identity-namespaced (`casmi.workspace.cache_identity`): a change of query set, regime table, hidden truths, reference library, universe or config selects a new namespace -- stale shards are never resumed. Fingerprints are cached with buffered writes (one shard per chunk).

In [ ]:
# >>> CASMI BOOTSTRAP casmi-v2-bootstrap-2 (generated by scripts/stamp_notebooks.py; do not edit by hand)
# mount Drive -> env roots -> clone if missing, else fetch + checkout + pull --ff-only -> drop stale casmi modules -> src on sys.path
import importlib, os, subprocess, sys
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', '/content/Enveda'))
os.environ.setdefault('ENVEDA_SCRATCH_DIR', '/content/enveda_work')
BRANCH = os.environ.setdefault('ENVEDA_BRANCH', 'main')
SYNC_REPO = os.environ.get('ENVEDA_SYNC_REPO', '1' if IN_COLAB else '0') == '1'  # outside Colab the working tree is used as is
GITHUB_REPO = 'mybenkhadda/enveda-kaggle'


def _git(*args):
    r = subprocess.run(['git', '-C', str(REPO_ROOT), *args], capture_output=True, text=True)
    if r.returncode != 0:
        cmd = ' '.join(args)
        raise RuntimeError(f'git {cmd} failed in {REPO_ROOT}: {r.stderr.strip()} | Fix: discard local edits inside the clone '
                           f'(git -C {REPO_ROOT} stash) or delete {REPO_ROOT}, then re-run this cell; source lives on GitHub only.')
    return r.stdout.strip()


if SYNC_REPO:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')  # optional Colab secret (private repository)
        except Exception:
            token = None
    url = f'https://{token}@github.com/{GITHUB_REPO}.git' if token else f'https://github.com/{GITHUB_REPO}.git'
    if not (REPO_ROOT / '.git').exists():
        r = subprocess.run(['git', 'clone', '--branch', BRANCH, url, str(REPO_ROOT)], capture_output=True, text=True)
        if r.returncode != 0:
            err = r.stderr.replace(token, '***') if token else r.stderr
            raise RuntimeError(f'git clone of {GITHUB_REPO} failed: {err.strip()} | Fix: check the GITHUB_TOKEN Colab secret / network.')
    else:
        _git('fetch', 'origin', BRANCH)
        _git('checkout', BRANCH)
        _git('pull', '--ff-only', 'origin', BRANCH)
if not (REPO_ROOT / 'src' / 'casmi' / 'workspace' / 'versions.py').exists():
    raise RuntimeError(f'{REPO_ROOT} has no src/casmi/workspace/versions.py: the source is OLDER than this notebook. '
                       f'Fix: push the bootstrap/artifact-registry refactor to GitHub {BRANCH}, restart the runtime, re-run this cell.')
for _m in [m for m in list(sys.modules) if m == 'casmi' or m.startswith('casmi.')]:
    del sys.modules[_m]  # after a pull: never mix old and new source in one kernel
importlib.invalidate_caches()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['ENVEDA_BOOTSTRAP_CELL'] = 'casmi-v2-bootstrap-2'
_head = _git('log', '-1', '--format=%h %s') if (REPO_ROOT / '.git').exists() else 'no git (uploaded copy)'
print(f'source {REPO_ROOT} @ {_head} | drive {DRIVE_ROOT} | colab {IN_COLAB} | synced {SYNC_REPO}')

In [ ]:
from casmi.workspace.bootstrap import CHEM_PACKAGES, RANK_PACKAGES, bootstrap

NOTEBOOK = '14_colab_analog_baseline'
ALLOW_PROTOCOL_INVALID_DIAGNOSTIC = False   # True: C1-only diagnostic without a valid Gate A (runs recorded as PRELIMINARY)
QUICK_RUN_MAX_QUERIES_PER_FOLD = 2000       # None = every query (deterministic whole-molecule subsample otherwise)
RUN_TAG = 'analog_v3' + ('_quick' if QUICK_RUN_MAX_QUERIES_PER_FOLD else '')
CTX = bootstrap(NOTEBOOK, notebook_api='casmi-v2-notebooks-4',
                uses_paths=('repo_root', 'bundle_dir', 'scratch_dir', 'experiments_dir'),
                uses_artifacts=('universe_root', 'analog_library_cache_dir', 'fingerprint_cache_dir', 'analog_neighbors_cache_dir',
                                'analog_features_dir', 'analog_reports_dir', 'analog_ranker_checkpoints_dir', 'validation_predictions_dir'),
                requires=('dev_queries', 'validation_regimes', 'validation_regimes_meta', 'bundle_config', 'bundle_ref_meta', 'bundle_connectivities',
                          'universe_manifest', 'candidate_keys', 'bucket_offsets', 'candidate_mass_index'),
                optional=('gate_a_decision',),
                packages={**CHEM_PACKAGES, **RANK_PACKAGES}, check_bundle=True,
                signatures={'casmi.analog.pipeline:run_analog_fold': ['fold', 'queries', 'regimes', 'alib', 'searcher', 'index', 'universe_root',
                                                                       'candidate_keys', 'fp_cache', 'cfg', 'neighbors_dir', 'features_dir',
                                                                       'identity_extra'],
                            'casmi.analog.pipeline:load_features': ['namespace_dirs'],
                            'casmi.ranking.lambdamart:fit_fold_models': ['train_df', 'feature_cols', 'eval_query_ids_by_fold'],
                            'casmi.ranking.molecule_fusion:evaluate_fusions': ['features', 'score_col', 'queries', 'methods'],
                            'casmi.ranking.fold_stability:decision_table': ['per_fold', 'baseline', 'metric'],
                            'casmi.validation.leakage_audit:audit_analog': ['queries', 'positives_per_query', 'disjointness', 'features',
                                                                            'score_cols', 'feature_cols', 'runs', 'regimes', 't2_policy'],
                            'casmi.workspace.experiments:log_experiment': ['experiments_dir', 'experiment_id', 'description', 'channels', 'notebook',
                                                                          'fold', 'status', 'decision', 'universe_identity', 'artifact_paths']})
CFG, P, ARTIFACTS, MON = CTX.cfg, CTX.paths, CTX.artifacts, CTX.monitor

## 1. Gate-A protocol check, regimes and queries

Gate A must be protocol-valid **and** computed for the current universe (identity match).

In [ ]:
import json, time
import numpy as np, pandas as pd
from casmi.validation.c2_protocol import assess_c2_protocol, assess_universe, gate_a_protocol_status
from casmi.workspace.artifact_registry import read_json

REG = pd.read_parquet(ARTIFACTS.validation_regimes)
REG_META = read_json(ARTIFACTS.validation_regimes_meta)
GATE_A = read_json(ARTIFACTS.gate_a_decision) or {'decision': 'NOT_RUN', 'protocol_valid': False}
UNIVERSE = assess_universe(ARTIFACTS, CFG)
PROTOCOL = assess_c2_protocol(UNIVERSE, REG_META, int((REG['regime'] == 'C2').sum()))
GATE_A_VALID = gate_a_protocol_status(GATE_A) and PROTOCOL['protocol_valid'] and GATE_A.get('universe_identity') == UNIVERSE.identity
RUN_STATUS = 'COMPLETED' if GATE_A_VALID else 'PRELIMINARY'
print('Gate A:', GATE_A.get('decision'), '| protocol valid now:', PROTOCOL['protocol_valid'], '| GATE_A_VALID:', GATE_A_VALID)
if not GATE_A_VALID:
    REASONS = PROTOCOL['reasons'] or ['Gate A is missing, protocol-invalid or was computed for another universe: ' + str(GATE_A.get('decision'))]
    if not ALLOW_PROTOCOL_INVALID_DIAGNOSTIC:
        raise RuntimeError('ERROR: notebook 14 requires a protocol-valid Gate A (notebook 13): ' + ' | '.join(REASONS)
                           + ' -- Fix: external source -> 12 -> 11 (REBUILD = True) -> 13; or set ALLOW_PROTOCOL_INVALID_DIAGNOSTIC = True'
                           + ' for a C1-only diagnostic (C2 numbers PRELIMINARY, excluded from the leaderboard).')
    print('*** DIAGNOSTIC RUN: C2 numbers are PRELIMINARY (shortcut risk); decisions use C1 only. Reasons:', REASONS)

In [ ]:
from casmi.validation.regimes import stable_unit_hash

dev = pd.read_parquet(ARTIFACTS.dev_queries, columns=['query_id', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type'])
dev['query_id'] = dev['query_id'].astype(str)
Q = REG.dropna(subset=['regime']).merge(dev, on='query_id', how='left', validate='one_to_one')
if QUICK_RUN_MAX_QUERIES_PER_FOLD:
    # subsample whole MOLECULES (truth connectivities), so molecule-level fusion sees every spectrum of a molecule
    Q['_h'] = stable_unit_hash(Q['true_connectivity_key'].tolist(), str(CFG['validation']['split_seed']) + '|quick')
    Q = Q.sort_values(['_h', 'query_id'])
    Q = Q[Q.groupby('fold').cumcount() < QUICK_RUN_MAX_QUERIES_PER_FOLD].drop(columns='_h')
Q = Q.sort_values(['fold', 'query_id']).reset_index(drop=True)
display(Q.groupby(['fold', 'regime']).size().unstack(fill_value=0))

## 2. Reference library (staged to the local SSD), universe, fingerprint cache, cache identity

In [ ]:
from casmi.analog.retrieval import AnalogConfig, AnalogLibrary, AnalogSearcher
from casmi.workspace.artifact_registry import bundle_identity
from casmi.workspace.staging import stage_directory

ACFG = AnalogConfig.from_dict(CFG['analog'])
BUNDLE_ID = bundle_identity(ARTIFACTS)
LIB_CACHE = ARTIFACTS.analog_library_cache_dir / (str(BUNDLE_ID['bundle_version']) + '_' + str(BUNDLE_ID['CONFIG_HASH']))
with MON.stage('stage + bin reference library'):
    # stage ONLY the reference-library files the search reads; the Drive bundle is never modified
    LIB_DIR, staged = stage_directory(P.bundle_dir, P.scratch_dir, rel='bundle_reflib',
                                      patterns=('ref_meta.parquet', 'ref_index_*.npy', 'ref_peaks_*.npy', 'connectivities.parquet'))
    LIB_DIR = P.bundle_dir if staged['not_staged'] else LIB_DIR
    ALIB = AnalogLibrary(LIB_DIR)
    rows = ALIB.rows_of_spectrum_ids(Q['query_id'])
    assert (rows >= 0).all(), 'dev query ids must be library (training) spectrum ids -- check the id mapping before continuing'
    LF, LN = ALIB.binned_matrices(ACFG, cache_dir=LIB_CACHE)
    SEARCHER = AnalogSearcher(ALIB, LF, LN, ACFG)
print(f'library rows: {ALIB.n_refs:,} | fragments nnz={LF.nnz:,} | neutral losses nnz={LN.nnz:,} | device={SEARCHER.device}')

In [ ]:
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.chemistry.fingerprint_store import FingerprintCache
from casmi.workspace.cache_identity import reference_library_identity
from casmi.workspace.staging import stage_file
from casmi.workspace.versions import NOTEBOOK_API_VERSION

IDX_DIR, staged = stage_directory(ARTIFACTS.candidate_mass_index, P.scratch_dir, rel='universe/index')
IDX = CandidateMassIndex.load(ARTIFACTS.candidate_mass_index if staged['not_staged'] else IDX_DIR)
UKEYS = np.load(stage_file(ARTIFACTS.candidate_keys, P.scratch_dir, rel='universe/candidate_keys.npy'), mmap_mode='r')
FP = FingerprintCache(ARTIFACTS.fingerprint_cache_dir, CFG['analog']['fingerprint']['radius'], CFG['analog']['fingerprint']['n_bits'])
IDENTITY_EXTRA = {'reference_library': reference_library_identity(BUNDLE_ID, ARTIFACTS.bundle_ref_meta), 'universe': UNIVERSE.identity,
                  'notebook_api': NOTEBOOK_API_VERSION}
print(f'universe candidates: {IDX.n_candidates:,} | cached fingerprints: {len(FP):,}')
print('cache identity extras:', json.dumps(IDENTITY_EXTRA, default=str))

## 3. Per-fold analog neighbors and candidate features (identity-namespaced, resumable)

In [ ]:
from casmi.analog.pipeline import load_features, run_analog_fold

NB_DIR = ARTIFACTS.analog_neighbors_cache_dir / RUN_TAG
FEAT_DIR = ARTIFACTS.analog_features_dir / RUN_TAG
RUNS = []
for f in sorted(Q['fold'].unique()):
    with MON.stage('analog fold ' + str(f), n_items=int((Q['fold'] == f).sum()), unit='queries'):
        t0 = time.time()
        rec = run_analog_fold(f, Q, REG, ALIB, SEARCHER, IDX, ARTIFACTS.universe_root, UKEYS, FP, CFG['analog'], NB_DIR, FEAT_DIR,
                              identity_extra=IDENTITY_EXTRA)
        rec['seconds'] = time.time() - t0
    RUNS.append(rec)
    print(rec)
FP.write_meta()
F = load_features([r['features_dir'] for r in RUNS])
F = F[F['query_id'].isin(set(Q['query_id']))].reset_index(drop=True)
pos = F.groupby('query_id')['is_true_candidate'].sum()
display(Q.assign(n_pos=Q['query_id'].map(pos).fillna(0)).groupby('regime')['n_pos'].agg(['mean', 'max']))
print(f'feature rows: {len(F):,}')

## 4. Heuristic (untrained) scorers, per spectrum

In [ ]:
from casmi.analog.pipeline import per_query_ranks
from casmi.validation.metrics import composite_mrr, ranking_metrics

F['score_mass_only'] = -F['abs_mass_error_ppm']
SCORERS = {'mass_only': 'score_mass_only', 'analog_propagation': 'analog_propagation_score', 'analog_max_weighted': 'analog_max_weighted'}
PER_QUERY, ROWS = {}, []
for name, col in SCORERS.items():
    F['score_' + name] = F[col]
    pq = per_query_ranks(F, col, Q)
    PER_QUERY[name] = pq
    for reg, g in pq.groupby('regime'):
        ROWS.append({'variant': name, 'kind': 'heuristic', 'regime': reg, **ranking_metrics(g)})
display(pd.DataFrame(ROWS))

## 5. LightGBM lambdarank baseline and ablation (out-of-fold; model f never sees fold-f connectivities)

In [ ]:
from casmi.analog.features import ANALOG_FEATURES, MASS_FEATURES, feature_schema
from casmi.ranking.lambdamart import LGBM_PARAMS, connectivity_disjointness, fit_fold_models, predict_by_fold, save_fold_models
from casmi.validation.c2_protocol import assert_no_train_provenance_features

REVIEW_COLUMNS = assert_no_train_provenance_features(MASS_FEATURES + ANALOG_FEATURES)   # raises on TRAIN-provenance features
VARIANTS = {'lgbm_mass_only': MASS_FEATURES, 'lgbm_analog_only': ANALOG_FEATURES, 'lgbm_mass+analog': MASS_FEATURES + ANALOG_FEATURES}
train_q = set(pos[pos > 0].index) & set(Q.loc[Q['regime'].isin(['C1', 'C2']), 'query_id'])
TRAIN = F[F['query_id'].isin(train_q)]
qmeta = Q[['query_id', 'fold', 'true_connectivity_key']]
DISJ = connectivity_disjointness(qmeta[qmeta['query_id'].isin(train_q)], qmeta)
eval_ids = {f: set(g['query_id']) for f, g in Q.groupby('fold')}
MODELS = {}
for name, feats in VARIANTS.items():
    with MON.stage('fit ' + name, n_items=len(TRAIN), unit='rows'):
        models, audit = fit_fold_models(TRAIN, feats, eval_query_ids_by_fold=eval_ids)
        F['score_' + name] = predict_by_fold(models, F, feats).to_numpy()
    MODELS[name] = (models, feats)
    pq = per_query_ranks(F, 'score_' + name, Q)
    PER_QUERY[name] = pq
    for reg, g in pq.groupby('regime'):
        ROWS.append({'variant': name, 'kind': 'lambdarank', 'regime': reg, **ranking_metrics(g)})
ABLATION = pd.DataFrame(ROWS)
display(ABLATION.pivot(index='variant', columns='regime', values='mrr_at_25').round(4))

## 6. Composite, per-fold stability and the KEEP / REJECT rule

PROVISIONAL composite weights; raw regime metrics stay primary. `decision_table` (fold-stability rule): KEEP only if the variant beats `mass_only` on the pooled delta **and** on all folds but one; a mean-only gain is NEEDS_MORE_EVIDENCE. Decisions use C2 under a valid protocol, C1 in a diagnostic run. No variant is declared best without this evidence.

In [ ]:
from casmi.ranking.fold_stability import decision_table
from casmi.validation.reporting import add_strata, stratified_table

W = CFG['validation']['composite_weights_PROVISIONAL']
COMPOSITE = []
for name, g in ABLATION.groupby('variant'):
    c = composite_mrr(dict(zip(g['regime'], g['mrr_at_25'])), W)
    COMPOSITE.append({'variant': name, 'composite_mrr_at_25_PROVISIONAL': c['composite_mrr_at_25'],
                      **{r + '_mrr25': val for r, val in c['regime_mrr_at_25'].items()}})
COMPOSITE = pd.DataFrame(COMPOSITE).sort_values('composite_mrr_at_25_PROVISIONAL', ascending=False)
display(COMPOSITE)
PER_FOLD = pd.DataFrame([{'variant': n, 'fold': f, 'regime': r, **ranking_metrics(g)}
                         for n, pq in PER_QUERY.items() for (f, r), g in pq.groupby(['fold', 'regime'])])
DECISION_REGIME = 'C2' if GATE_A_VALID else 'C1'
DECISIONS = decision_table(PER_FOLD[PER_FOLD['regime'] == DECISION_REGIME], 'mass_only', metric='mrr_at_25')
display(DECISIONS)
DECISION_OF = dict(zip(DECISIONS['variant'], DECISIONS['decision']))
best = COMPOSITE['variant'].iloc[0]
STRAT = stratified_table(add_strata(PER_QUERY[best].rename(columns={'n_candidates': 'pool_size'}), Q, CFG['validation']['mass_bins_da'],
                                    CFG['validation']['pool_size_bins']), ranking_metrics, min_queries=20)
display(STRAT[STRAT['regime'] == DECISION_REGIME].head(40))

## 7. Molecule-level evaluation (late fusion, compared strategies)

The competition ranks **molecules**: a development molecule is the set of query spectra of one truth connectivity in one fold. Strategies compared (`casmi.ranking.molecule_fusion`): mean, max, log-sum-exp (several temperatures), reciprocal-rank fusion, top-k vote. **Caveat:** molecule-level C1 is optimistic (sibling query spectra remain references for each other under the per-query C1 protocol); C2 / C3 are unaffected.

In [ ]:
from casmi.ranking.molecule_fusion import evaluate_fusions

MOL_ROWS, MOL_PER = [], {}
for name in dict.fromkeys(['mass_only', 'analog_propagation', best]):
    with MON.stage('fusion ' + name):
        per_mol = evaluate_fusions(F, 'score_' + name, Q)
    MOL_PER[name] = per_mol
    for (fu, reg), g in per_mol.groupby(['fusion', 'regime']):
        MOL_ROWS.append({'variant': name, 'fusion': fu, 'regime': reg, **ranking_metrics(g)})
MOLECULE = pd.DataFrame(MOL_ROWS)
display(MOLECULE.pivot_table(index=['variant', 'fusion'], columns='regime', values='mrr_at_25').round(4))
print('molecules per regime:', MOL_PER[best].drop_duplicates('molecule_id').groupby('regime').size().to_dict(),
      '| spectra per molecule (median):', float(MOL_PER[best]['n_spectra'].median()))

In [ ]:
from casmi.viz import plots_v2

OUT = ARTIFACTS.analog_reports_dir / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
plots_v2.channel_ablation(ABLATION[ABLATION['regime'].isin(['C1', 'C2'])], metric='mrr_at_25').savefig(OUT / 'ablation_mrr25.png', dpi=120)
plots_v2.channel_ablation(ABLATION[ABLATION['regime'].isin(['C1', 'C2'])], metric='hit_at_1').savefig(OUT / 'ablation_hit1.png', dpi=120)
plots_v2.truth_rank_distribution(PER_QUERY[best]).savefig(OUT / 'truth_rank.png', dpi=120)

## 8. LEAKAGE AUDIT

`casmi.validation.leakage_audit.audit_analog` raises on any failed assertion: C3 truths are never candidates; at most one positive per query; ranker training / evaluation connectivity-disjoint per fold (fold isolation); every score out-of-fold; no TRAIN-provenance / reference-availability ranking feature; each fold's feature cache identity equals the current hidden sets and query set (no stale cache after a regime change). `run_analog_fold` additionally asserts per fold that no hidden connectivity appears as an analog and that no query matches its own spectrum. **Documented limitation:** T1 identity duplicates are excluded, T2 duplicates are not (identity peaks are not shipped in the bundle) -- reported as a limitation row, never as a passed check. Queries are training spectra; the visible test set is never used.

In [ ]:
from casmi.analog.retrieval import T2_POLICY
from casmi.validation.leakage_audit import audit_analog

SCORE_COLS = ['score_' + n for n in VARIANTS]
AUDIT = audit_analog(Q, pos, DISJ, F, SCORE_COLS, MASS_FEATURES + ANALOG_FEATURES, RUNS, REG, T2_POLICY)
display(AUDIT)
print('provenance-correlated feature columns needing an ablation before use:', REVIEW_COLUMNS)

## 9. Persist

In [ ]:
from casmi.ranking.rank_eval import rank_by_keys
from casmi.validation.reporting import save_report
from casmi.workspace.artifacts import write_metadata

CKPT = ARTIFACTS.analog_ranker_checkpoints_dir / RUN_TAG
for name, (models, feats) in MODELS.items():
    save_fold_models(models, CKPT / name, feats, LGBM_PARAMS, sorted(train_q),
                     extra={'notebook': NOTEBOOK, 'feature_schema': feature_schema(feats), 'regime_signature': REG_META.get('signature')})
PRED = ARTIFACTS.validation_predictions_dir / 'analog_baseline' / RUN_TAG
PRED.mkdir(parents=True, exist_ok=True)
r = rank_by_keys(F[['query_id', 'fold', 'candidate_id', 'connectivity_key', 'is_true_candidate', 'abs_mass_error_ppm', 'score_' + best]],
                 [('score_' + best, False), ('abs_mass_error_ppm', True), ('candidate_id', True)])
r[r['rank'] <= 25].to_parquet(PRED / (best + '_top25.parquet'), index=False)
for name, pm in MOL_PER.items():
    pm.to_parquet(PRED / (name + '_molecule_ranks.parquet'), index=False)
save_report(ABLATION, {'notebook': NOTEBOOK, 'run_tag': RUN_TAG, 'run_status': RUN_STATUS, 'quick_run_cap': QUICK_RUN_MAX_QUERIES_PER_FOLD,
                       'gate_a': GATE_A, 'protocol': PROTOCOL, 'composite_PROVISIONAL': COMPOSITE.to_dict(orient='records'), 'runs': RUNS,
                       'decisions': DECISIONS.to_dict(orient='records'), 'molecule_level': MOLECULE.to_dict(orient='records'),
                       'leakage_audit': AUDIT.to_dict(orient='records'), 'resources': MON.table().to_dict(orient='records')},
            OUT, 'analog_baseline_metrics')
STRAT.to_parquet(OUT / 'stratified.parquet', index=False)
PER_FOLD.to_parquet(OUT / 'per_fold.parquet', index=False)
MOLECULE.to_parquet(OUT / 'molecule_level.parquet', index=False)
write_metadata(OUT, NOTEBOOK, config={'analog': CFG['analog']}, seed=CTX.seed, repo_root=P.repo_root, model_name='lightgbm_lambdarank',
               feature_list=MASS_FEATURES + ANALOG_FEATURES, schema_version=feature_schema()['schema_version'],
               input_paths={'regimes': ARTIFACTS.validation_regimes, 'bundle': P.bundle_dir, 'universe': ARTIFACTS.universe_root},
               extra=CTX.metadata(run_status=RUN_STATUS, cache_identity_extra=IDENTITY_EXTRA))

## 10. Append-only experiment records

One pooled + one per-fold run per variant (spectrum level) and one run per (variant, fusion) at molecule level. Diagnostic runs (no valid Gate A) carry `status = PRELIMINARY`.

In [ ]:
from casmi.workspace.experiments import log_experiment


def _metrics(g):
    m = ranking_metrics(g)
    return {'MRR25': m['mrr_at_25'], 'Top1': m['hit_at_1'], 'Top5': m['hit_at_5'], 'Top10': m['hit_at_10'], 'Top25': m['hit_at_25'],
            'mean_rank': m['mean_truth_rank'], 'median_rank': m['median_truth_rank'], 'candidate_recall': 1 - m['miss_rate'], 'n_queries': m['n_queries']}


common = dict(config_path=CFG['_config_path'], git_commit=CTX.git_commit, notebook=NOTEBOOK, config=CFG['analog'], seed=CTX.seed,
              candidate_generator='mass_index_' + str(CFG['analog']['candidate_ppm']) + 'ppm', status=RUN_STATUS,
              regime_identity=REG_META.get('signature'), universe_identity=UNIVERSE.identity, notes=RUN_TAG,
              artifact_paths={'reports': OUT, 'predictions': PRED, 'checkpoints': CKPT})
for _, c in COMPOSITE.iterrows():
    name, pq = c['variant'], PER_QUERY[c['variant']]
    log_experiment(P.experiments_dir, RUN_TAG + ':' + name, 'analog baseline, spectrum level (notebook 14)', ['mass', 'analog'], model=name,
                   ranker='lambdarank' if name.startswith('lgbm') else 'heuristic', features=VARIANTS.get(name, [SCORERS.get(name)]),
                   decision=DECISION_OF.get(name), C1_MRR25=c.get('C1_mrr25'), C2_MRR25=c.get('C2_mrr25'), C3_MRR25=c.get('C3_mrr25'),
                   composite_MRR25=c['composite_mrr_at_25_PROVISIONAL'], runtime=MON.elapsed(), **_metrics(pq), **common)
    for f, g in pq.groupby('fold'):
        log_experiment(P.experiments_dir, RUN_TAG + ':' + name, 'analog baseline, spectrum level, one fold', ['mass', 'analog'], model=name,
                       fold=int(f), **{reg + '_MRR25': ranking_metrics(gg)['mrr_at_25'] for reg, gg in g.groupby('regime')}, **_metrics(g), **common)
for (name, fu), g in MOLECULE.groupby(['variant', 'fusion']):
    regs = dict(zip(g['regime'], g['mrr_at_25']))
    log_experiment(P.experiments_dir, RUN_TAG + ':' + name + ':molecule:' + fu, 'analog baseline, MOLECULE level (late fusion)', ['mass', 'analog'],
                   model=name, ranker='late_fusion', features=fu, decision='DIAGNOSTIC', composite_MRR25=composite_mrr(regs, W)['composite_mrr_at_25'],
                   **{reg + '_MRR25': val for reg, val in regs.items()}, **_metrics(MOL_PER[name][MOL_PER[name]['fusion'] == fu]), **common)
print('saved under', OUT, '|', PRED, '|', CKPT, '| run status', RUN_STATUS)